In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, train_test_split
from sklearn.metrics import classification_report, f1_score

# Rutas
INPUT_PATH = '../../data/processed/dataset_refrigeracion_processed_not_scaled.csv'
MODEL_PATH = '../../models/artifacts/'


if not os.path.exists(MODEL_PATH):
    os.makedirs(MODEL_PATH)

In [2]:
df = pd.read_csv(INPUT_PATH)

# Definir columnas a ignorar
target_column = 'fault_id'
drop_cols = [target_column, 'fault', 'run_id', 'time_min']

# Separar X e y
X = df.drop(columns=drop_cols)
y = df[target_column]
groups = df['run_id']

print(f"Dataset cargado con {X.shape[1]} características.")
print(f"Distribución de clases:\n{y.value_counts(normalize=True).head()}")

Dataset cargado con 72 características.
Distribución de clases:
fault_id
0    0.076923
1    0.076923
2    0.076923
3    0.076923
4    0.076923
Name: proportion, dtype: float64


In [3]:
# Obtenemos un sello de cada run para hacer el split estratificado
run_labels = df.groupby('run_id')[target_column].first()
unique_runs = run_labels.index

train_runs, test_runs = train_test_split(
    unique_runs, 
    test_size=0.2, 
    random_state=42,
    stratify=run_labels 
)

train_df = df[df['run_id'].isin(train_runs)].copy()

# Ahora sí, definimos X_train e y_train como antes
X_train = train_df.drop(columns=drop_cols)
y_train = train_df[target_column]
groups_train = train_df['run_id']

# Hacemos lo mismo para test por coherencia
test_df = df[df['run_id'].isin(test_runs)].copy()
X_test = test_df.drop(columns=drop_cols)
y_test = test_df[target_column]

print(f"Entrenamiento: {len(train_runs)} ciclos | Test: {len(test_runs)} ciclos")

Entrenamiento: 1040 ciclos | Test: 260 ciclos


In [4]:
from sklearn.preprocessing import StandardScaler

# 1. Identificar columnas numéricas de sensores
exclude = ['run_id', 'time_min', 'fault_id', 'fault', 'defrost_on', 'door_open']
numerical_cols = [c for c in X_train.columns if c not in exclude]

# 2. Escalar: Fit solo en Train para evitar Data Leakage
scaler = StandardScaler()
X_train_scaled_vals = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled_vals = scaler.transform(X_test[numerical_cols])

# 3. Guardar el objeto scaler (vital para la evaluación posterior)
joblib.dump(scaler, os.path.join(MODEL_PATH, 'scaler_refrigeracion.pkl'))

# 4. Reconstrucción de DataFrames para el CSV Maestro
train_final = train_df.copy()
test_final = test_df.copy()

train_final[numerical_cols] = X_train_scaled_vals
test_final[numerical_cols] = X_test_scaled_vals

# Unimos y guardamos el dataset completo con metadatos
df_final_procesado = pd.concat([train_final, test_final], axis=0).sort_index()
output_path = '../../data/processed/dataset_refrigeracion_processed.csv'
df_final_procesado.to_csv(output_path, index=False)

# --- EL AJUSTE CLAVE ---
# 5. Definir las variables que usará la celda de Tuning (sin columnas de texto)
X_train_scaled = train_final.drop(columns=drop_cols)
X_test_scaled = test_final.drop(columns=drop_cols)

print(f"✅ Scaler guardado y dataset completo exportado a: {output_path}")

✅ Scaler guardado y dataset completo exportado a: ../../data/processed/dataset_refrigeracion_processed.csv


In [5]:
# Celda 3.5: Definición de Pesos Basados en Física
from sklearn.utils.class_weight import compute_class_weight

def compute_sample_weights(df_phys, target_col='fault_id'):
    # Pesado por desbalance de clases (Balanced)
    classes = np.unique(df_phys[target_col])
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=df_phys[target_col])
    class_weight_dict = dict(zip(classes, class_weights))
    
    weights = df_phys[target_col].map(class_weight_dict).values.copy()

    # Bonus de importancia para casos críticos (Incondensables y Condensador Sucio)
    p_dis_threshold = df_phys["mean_P_dis_bar"].quantile(0.90)
    early_error_threshold = df_phys["early_P_dis_error"].quantile(0.90)

    # Si hay mucho error de presión al inicio, el modelo DEBE aprender a no fallar ahí
    weights *= np.where(df_phys["early_P_dis_error"] > early_error_threshold, 1.5, 1.0)
    weights *= np.where(df_phys["mean_P_dis_bar"] > p_dis_threshold, 1.2, 1.0)
    
    return weights


# Aplicar solo al set de entrenamiento
train_sample_weights = compute_sample_weights(train_df, target_col='fault_id')

In [6]:
# Espacio de búsqueda optimizado para evitar modelos excesivamente pesados
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [15, 20, 25],
    'min_samples_leaf': [10, 20, 50],
    'max_features': ['sqrt'],
    'bootstrap': [True]
}

# Validación cruzada respetando grupos y estratos
cv_strategy = StratifiedGroupKFold(n_splits=3)

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=5, # Aumentar si se dispone de más tiempo
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

In [7]:
print("--- INICIANDO TUNING PARA REFRIGERACIÓN ---")

rf_search.fit(X_train_scaled, y_train, groups=groups_train, sample_weight=train_sample_weights)

print("\n Tuning completado.")
print(f"Mejor F1-Score obtenido en CV: {rf_search.best_score_:.4f}")

--- INICIANDO TUNING PARA REFRIGERACIÓN ---
Fitting 3 folds for each of 5 candidates, totalling 15 fits

 Tuning completado.
Mejor F1-Score obtenido en CV: 0.9176


In [8]:
# Mostrar mejores parámetros
best_params = rf_search.best_params_
print("MEJORES HIPERPARÁMETROS ENCONTRADOS:")
for p, v in best_params.items():
    print(f" - {p}: {v}")

# Guardar el modelo
best_rf = rf_search.best_estimator_
joblib.dump(best_rf, os.path.join(MODEL_PATH, 'rf_refrigeracion_tuned.pkl'))

# Guardar el esquema de features 
import json
schema = {"features": list(X_train.columns)}
with open(os.path.join(MODEL_PATH, 'feature_schema_refrigeracion.json'), 'w') as f:
    json.dump(schema, f)

print(f"\n Artefactos guardados en {MODEL_PATH}")

MEJORES HIPERPARÁMETROS ENCONTRADOS:
 - n_estimators: 100
 - min_samples_leaf: 10
 - max_features: sqrt
 - max_depth: 15
 - bootstrap: True

 Artefactos guardados en ../../models/artifacts/


In [9]:
y_pred = best_rf.predict(X_test_scaled)
print("--- REPORTE DE RENDIMIENTO INICIAL (TEST) ---")
print(classification_report(y_test, y_pred))

--- REPORTE DE RENDIMIENTO INICIAL (TEST) ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     28620
           1       0.96      0.97      0.97     28620
           2       0.44      0.50      0.47     28620
           3       1.00      1.00      1.00     28620
           4       1.00      1.00      1.00     28620
           5       0.99      1.00      1.00     28620
           6       1.00      1.00      1.00     28620
           7       1.00      1.00      1.00     28620
           8       1.00      1.00      1.00     28620
           9       1.00      1.00      1.00     28620
          10       1.00      1.00      1.00     28620
          11       0.42      0.35      0.38     28620
          12       1.00      1.00      1.00     28620

    accuracy                           0.91    372060
   macro avg       0.91      0.91      0.91    372060
weighted avg       0.91      0.91      0.91    372060

